# Reviewer-response evidence

This notebook verifies the unchanged input cohort, compares corrected coverage with pinned main, and exports the two editable response tables. It uses the existing first-order engine. No density assignments, observation filters, Monte Carlo runs, or correlation experiments are added.

Run the paper diagrams, weight, loss, D11, and hand-verification notebooks first. The baseline executes archived source in a separate Python process with the same installed dependencies and data.

In [1]:
from pathlib import Path
import sys
import time
repo_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/snowpyt_mechparams").is_dir())
for folder in ["src", "examples", "paper_analysis"]:
    sys.path.insert(0, str(repo_root / folder))
from review_metrics import write_json
results_dir = repo_root / "paper_analysis/reviewer_results"
results_dir.mkdir(parents=True, exist_ok=True)
notebook_started = time.perf_counter()

import hashlib
import importlib.metadata
import io
import json
import os
import platform
import subprocess
import tarfile
import tempfile
import warnings
import pandas as pd
from review_audit import load_audited_pits
from review_metrics import calculate_targets, coverage_summary, d11_comparison
warnings.filterwarnings("ignore", message="Using UFloat objects with std_dev==0.*")
BASELINE = "c8b712e93fe219c9c5dd71e6a5a8a3a6d947866c"


## Cohort and exact-density audit
Counts describe the retained observations; anomalies and duplicate records are reported without filtering.

In [2]:
pits, audit = load_audited_pits(repo_root / "examples/data")
slabs = [s for p in pits for s in p.create_slabs("ECTP_failure_layer")]
expected = {"files": 50278, "layers": 371429, "slabs": 14776,
            "contributing_pits": 12347, "pits_with_multiple_slabs": 2113,
            "density_observations": 19118, "density_observations_exact": 10468,
            "observations_with_unusable_interval": 8650,
            "point_observations_located_in_layer": 8481,
            "unmatched_with_usable_contained_interval": 0}
assert all(audit[k] == v for k, v in expected.items())
assert audit["slab_exclusions"] == {"failure_depth_outside_layers": 283, "no_overlying_layer": 197}
write_json(results_dir / "audit_summary.json", audit)
display(pd.Series(audit))


files                                                                                   50278
parsed_pits                                                                             50278
layers                                                                                 371429
slabs                                                                                   14776
contributing_pits                                                                       12347
pits_with_multiple_slabs                                                                 2113
duplicate_file_content_excess                                                              16
duplicate_pit_id_excess                                                                    16
repeated_stratigraphies_within_pit                                                       1265
issues                                      {'layer_overlap': 163, 'layer_gap': 173, 'meas...
slab_exclusions                             {'failure_depth_

## Independent current coverage
Each target retains all 14,776 starting slabs in its denominator. The elasticity target checks availability of the additional layer inputs; it does not change shear weight.

In [3]:
started = time.perf_counter()
records = calculate_targets(slabs)
coverage, coverage_table = coverage_summary(records)
current_d11, per_slab, common = d11_comparison(records)
current_seconds = time.perf_counter() - started
assert {x["target"]: x["union"] for x in coverage} == {
    "slab_weight_shear": 5535, "slab_weight_shear_with_elasticity": 772, "D11": 823}
assert current_d11["common_slabs"] == 316
# Independently calculated D11 values and SDs must match the main D11 notebook.
expected_d11 = pd.read_csv(results_dir / "d11_common.csv").set_index(["slab_id", "pathway"])
actual_d11 = common.set_index(["slab_id", "pathway"]).reindex(expected_d11.index)
import numpy as np
np.testing.assert_allclose(actual_d11[["value", "sigma"]], expected_d11[["value", "sigma"]], rtol=1e-12)
coverage_table.to_csv(results_dir / "coverage.csv", index=False)
write_json(results_dir / "coverage_summary.json", coverage)
display(coverage_table)


,target,pathway,successful_slabs,attempted_slabs,coverage_percent
22,D11,kim_jamieson_table2 -> wautier -> kochle,735,14776,4.974283
14,D11,geldsetzer -> wautier -> kochle,732,14776,4.953979
10,D11,geldsetzer -> kochle -> kochle,725,14776,4.906605
12,D11,geldsetzer -> schottner -> kochle,694,14776,4.696806
18,D11,kim_jamieson_table2 -> kochle -> kochle,626,14776,4.236600
...,...,...,...,...,...
63,slab_weight_shear_with_elasticity,kim_jamieson_table6 -> kochle -> srivastava,5,14776,0.033839
65,slab_weight_shear_with_elasticity,kim_jamieson_table6 -> schottner -> srivastava,5,14776,0.033839
67,slab_weight_shear_with_elasticity,kim_jamieson_table6 -> wautier -> srivastava,5,14776,0.033839
60,slab_weight_shear_with_elasticity,kim_jamieson_table6 -> bergfeld -> kochle,0,14776,0.000000


## Pinned baseline comparison
The archived baseline is numerical source, not a runtime override of corrected functions. Only compact comparison summaries are retained. The historical saved notebook reported 317 common slabs; fresh baseline and corrected runs both yield 316.

In [4]:
baseline_code = """
import json, sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore', message='Using UFloat objects with std_dev==0.*')
from snowpyt_mechparams.models import Pit
from snowpyt_mechparams.snowpilot import parse_caaml_directory
from review_metrics import calculate_targets, coverage_summary, d11_comparison
pits = [Pit.from_snow_pit(p) for p in parse_caaml_directory(sys.argv[1])]
slabs = [s for p in pits for s in p.create_slabs('ECTP_failure_layer')]
records = calculate_targets(slabs)
coverage, table = coverage_summary(records)
d11, per_slab, common = d11_comparison(records)
print(json.dumps({'coverage':table.to_dict('records'), 'common_slabs':d11['common_slabs'],
                  'common_slab_ids':sorted(per_slab.slab_id), 'pathways':sorted(d11['pathways'])}))
"""
started = time.perf_counter()
with tempfile.TemporaryDirectory(prefix="reviewer-baseline-") as temporary:
    archive = subprocess.check_output(["git", "archive", BASELINE, "src"], cwd=repo_root)
    with tarfile.open(fileobj=io.BytesIO(archive)) as source:
        source.extractall(temporary, filter="data")
    env = dict(os.environ)
    env["PYTHONPATH"] = os.pathsep.join([str(Path(temporary) / "src"), str(repo_root / "paper_analysis")])
    result = subprocess.run([sys.executable, "-c", baseline_code, str(repo_root / "examples/data")],
                            cwd=temporary, env=env, text=True, capture_output=True, check=True)
    baseline = json.loads(result.stdout)
baseline_seconds = time.perf_counter() - started
assert baseline["common_slabs"] == 316
assert baseline["common_slab_ids"] == sorted(per_slab.slab_id)
assert baseline["pathways"] == sorted(current_d11["pathways"])
comparison = pd.DataFrame(baseline["coverage"]).merge(coverage_table, on=["target", "pathway"], suffixes=("_baseline", "_corrected"), validate="one_to_one")
comparison["delta"] = comparison.successful_slabs_corrected - comparison.successful_slabs_baseline
affected = comparison[comparison.delta.ne(0)].copy()
assert all("wautier" in p or "schottner" in p for p in affected.pathway)
assert affected.loc[affected.pathway.str.contains("wautier"), "delta"].gt(0).all()
assert affected.loc[affected.pathway.str.contains("schottner"), "delta"].lt(0).all()
comparison.to_csv(results_dir / "coverage_comparison.csv", index=False)
write_json(results_dir / "baseline_comparison.json", {
    "baseline_commit": BASELINE, "baseline_common_slabs": baseline["common_slabs"],
    "corrected_common_slabs": current_d11["common_slabs"], "common_subset_unchanged": True,
    "top16_pathway_set_unchanged": True, "historical_saved_notebook_common_slabs": 317,
    "baseline_seconds": baseline_seconds, "current_calculation_seconds": current_seconds})
display(affected[["target", "pathway", "successful_slabs_baseline", "successful_slabs_corrected", "delta"]])


,target,pathway,successful_slabs_baseline,successful_slabs_corrected,delta
0,D11,kim_jamieson_table2 -> schottner -> kochle,735,625,-110
2,D11,geldsetzer -> schottner -> kochle,732,694,-38
8,D11,geldsetzer -> schottner -> srivastava,448,421,-27
11,D11,kim_jamieson_table2 -> wautier -> srivastava,445,450,5
16,D11,data_flow -> schottner -> kochle,17,14,-3
19,D11,data_flow -> schottner -> srivastava,12,11,-1
36,slab_weight_shear_with_elasticity,kim_jamieson_table2 -> schottner -> kochle,687,580,-107
38,slab_weight_shear_with_elasticity,geldsetzer -> schottner -> kochle,684,647,-37
44,slab_weight_shear_with_elasticity,geldsetzer -> schottner -> srivastava,418,393,-25
47,slab_weight_shear_with_elasticity,kim_jamieson_table2 -> wautier -> srivastava,415,420,5


## Editable response tables
Schöttner bounds are calibration-sample limits supplied by the author, not established predictive accuracy. The grain-size SD is a symmetric modeling approximation; it does not reproduce the asymmetric literature percentiles.

In [5]:
from snowpyt_mechparams.methods.layer._applicability import ELASTIC_DOMAINS
applicability = pd.DataFrame([
    {"Parameter": "E", "Method": method.title(), "Parent grain": grain,
     "Density minimum kg/m3": low, "Density maximum kg/m3": high,
     "Qualification": ("Author-supplied volume fractions times 916.7; equation uses 917" if method == "schottner" else
                       "MF inclusion is an implementation choice; source reports distinct MF behavior" if method == "kochle" and grain == "MF" else
                       "Source domain mapped to two-character parent class")}
    for method, domains in ELASTIC_DOMAINS.items() for grain, (low, high) in domains.items()
] + [
    {"Parameter": "nu", "Method": "Köchle", "Parent grain": grain,
     "Density minimum kg/m3": None, "Density maximum kg/m3": None,
     "Qualification": "No numerical density limit implemented; source reports no density correlation"}
    for grain in ["RG", "FC", "DH"]
] + [
    {"Parameter": "nu", "Method": "Srivastava", "Parent grain": grain,
     "Density minimum kg/m3": 200, "Density maximum kg/m3": 580 if grain == "RG" else None,
     "Qualification": "Lower density bound is strict (>200); RG upper bound inclusive"}
    for grain in ["PP", "DF", "RG", "FC", "DH"]
])
uncertainty = pd.DataFrame([
    ("Density", "10% of magnitude", "Assigned symmetric SD", "Conger and McClung (2009); Proksch et al. (2016)"),
    ("Slope angle", "2 degrees", "Assigned symmetric SD", "McCammon (2023) and observational experience"),
    ("Hand-hardness index", "0.67 (two subclasses)", "Assigned symmetric SD", "Observational experience; Geldsetzer index mapping"),
    ("Grain size", "48% of magnitude", "Assigned symmetric SD; not -48%/+122% percentiles", "Langlois et al. (2020), lower relative percentile width"),
    ("Layer thickness", "5% of magnitude", "Assigned symmetric SD", "Observational experience"),
], columns=["Observation", "Uncertainty", "Meaning", "Basis"])
table_dir = repo_root / "paper/tables"
for name, frame in [("elastic_applicability", applicability), ("observation_uncertainty", uncertainty)]:
    frame.to_csv(table_dir / f"{name}.csv", index=False)

display(applicability)
display(uncertainty)

# Group identical grain-specific intervals for a manuscript-width editable table.
def latex_text(value):
    return str(value).replace("%", r"\%").replace("&", r"\&").replace("_", r"\_")
rows = []
for method, domains in ELASTIC_DOMAINS.items():
    intervals = {}
    for grain, limits in domains.items():
        intervals.setdefault(limits, []).append(grain)
    for (low, high), grains in intervals.items():
        label = {"kochle": r"K\"ochle", "schottner": r"Sch\"ottner"}.get(method, method.title())
        rows.append(f"$E$ & {label} & {', '.join(grains)} & {low:.4f}--{high:.4f}" + r" \\")
rows.extend([
    r"$\nu$ & K\"ochle & RG, FC, DH & No numerical bound \\",
    r"$\nu$ & Srivastava & PP, DF, FC, DH & $>200$ \\",
    r"$\nu$ & Srivastava & RG & $(200,580]$ \\",
])
header = r"""\begin{tabularx}{\linewidth}{@{}llXl@{}}
\toprule
Parameter & Method & Parent grain forms & Density (kg\,m$^{-3}$) \\
\midrule
"""
notes = r"""\bottomrule
\end{tabularx}

\smallskip
{\footnotesize Elastic-modulus bounds are inclusive and evaluated at nominal density.
Subclasses map to two-character parents. Sch\"ottner bounds describe observed
calibration samples supplied by the author: volume fractions multiplied by
916.7 kg\,m$^{-3}$; equation normalization remains 917 kg\,m$^{-3}$.
K\"ochle's MF inclusion is an implementation choice: the source reports distinct
MF behavior without a separate fit. K\"ochle's Poisson ratio has no implemented
numerical density bound. These domains do not establish predictive accuracy.}
"""
(table_dir / "elastic_applicability.tex").write_text(header + "\n".join(rows) + "\n" + notes)
rows = [" & ".join(latex_text(x) for x in [r.Observation, r.Uncertainty, r.Basis]) + r" \\" for r in uncertainty.itertuples()]
header = r"""\begin{tabularx}{\linewidth}{@{}lp{0.23\linewidth}X@{}}
\toprule
Observation & Assigned symmetric SD & Basis \\
\midrule
"""
notes = r"""\bottomrule
\end{tabularx}

\smallskip
{\footnotesize All values are assigned standard deviations for first-order
measurement-uncertainty propagation. Grain-size SD is a symmetric 48\% modeling
approximation, not the asymmetric $-48\%/+122\%$ source percentile interval.
Parameterization uncertainty and grain-form misclassification are excluded.
Distinct layer observations are treated as independent.}
"""
(table_dir / "observation_uncertainty.tex").write_text(header + "\n".join(rows) + "\n" + notes)


,Parameter,Method,Parent grain,Density minimum kg/m3,Density maximum kg/m3,Qualification
0,E,Bergfeld,PP,110.0000,363.0000,Source domain mapped to two-character parent c...
1,E,Bergfeld,DF,110.0000,363.0000,Source domain mapped to two-character parent c...
2,E,Bergfeld,RG,110.0000,363.0000,Source domain mapped to two-character parent c...
3,E,Kochle,RG,150.0000,450.0000,Source domain mapped to two-character parent c...
4,E,Kochle,FC,150.0000,450.0000,Source domain mapped to two-character parent c...
5,E,Kochle,DH,150.0000,450.0000,Source domain mapped to two-character parent c...
6,E,Kochle,MF,150.0000,450.0000,MF inclusion is an implementation choice; sour...
7,E,Wautier,PP,103.0000,544.0000,Source domain mapped to two-character parent c...
8,E,Wautier,DF,103.0000,544.0000,Source domain mapped to two-character parent c...
9,E,Wautier,RG,103.0000,544.0000,Source domain mapped to two-character parent c...


,Observation,Uncertainty,Meaning,Basis
0,Density,10% of magnitude,Assigned symmetric SD,Conger and McClung (2009); Proksch et al. (2016)
1,Slope angle,2 degrees,Assigned symmetric SD,McCammon (2023) and observational experience
2,Hand-hardness index,0.67 (two subclasses),Assigned symmetric SD,Observational experience; Geldsetzer index map...
3,Grain size,48% of magnitude,Assigned symmetric SD; not -48%/+122% percentiles,"Langlois et al. (2020), lower relative percent..."
4,Layer thickness,5% of magnitude,Assigned symmetric SD,Observational experience


938

## Reproduction metadata
The dataset is tracked in Git. Its digest is SHA-256 over sorted `filename<TAB>file_sha256<NEWLINE>` entries. Source hashes identify the exact analysis state even before the final evidence commit.

In [6]:
sources = [*sorted((repo_root / "src").rglob("*.py")),
           *sorted((repo_root / "paper_analysis").glob("*.py")),
           *sorted((repo_root / "examples").glob("*.py"))]
metadata = {
    "baseline_commit": BASELINE,
    "source_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo_root, text=True).strip(),
    "input_git_tree": subprocess.check_output(["git", "rev-parse", "HEAD:examples/data"], cwd=repo_root, text=True).strip(),
    "input_manifest_sha256": audit["input_manifest_sha256"],
    "python": platform.python_version(), "platform": platform.platform(),
    "dependencies": {p: importlib.metadata.version(p) for p in ["numpy", "pandas", "scipy", "uncertainties", "snowpylot", "matplotlib"]},
    "source_sha256": {str(p.relative_to(repo_root)): hashlib.sha256(p.read_bytes()).hexdigest() for p in sources},
    "notebook_source_sha256": {str(p.relative_to(repo_root)): hashlib.sha256("\n".join("".join(c["source"]) for c in json.loads(p.read_text())["cells"]).encode()).hexdigest()
                              for p in [repo_root / f for f in ["examples/paper_diagrams.ipynb", "examples/slab_weight_inputs.ipynb", "examples/D11_analysis.ipynb", "paper_analysis/loss-analysis.ipynb", "paper_analysis/hand-verification.ipynb", "paper_analysis/reviewer_evidence.ipynb"]]},
    "seconds": time.perf_counter() - notebook_started,
}
write_json(results_dir / "reproduction.json", metadata)
print("Cohort, baseline comparison, and notebook D11 reconciliation passed.")


Cohort, baseline comparison, and notebook D11 reconciliation passed.
